# Signed Jumps 02a — the perp price path after a jump

Protocol: `discussions/Signed_Jumps/003_claude_sj02a_preregistration.md` (fixed before this run).

SJ 01 jumps (unchanged engine, no refit; model (e) parameters from `01/01_fits.csv`) → cluster-starter events → signed perp returns at 1–60 min, immediate and delayed entry → rule chosen on DEV by net lower bound → verdict on TEST.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_events as se
for m in (sj,se):importlib.reload(m)
assert sj.VERSION=='SJ01-0.1.0' and se.VERSION=='SJ02a-0.1.0'
SYNTHETIC=os.getenv('SJ02_SYNTHETIC','0')=='1'
p=sj.Params();ep=se.EventParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj02-');OUT=Path(tmp.name)/'02'
    p=sj.Params(n_boot=200,half_lives=(1,3),min_test_jumps_per_sign=20);ep=se.EventParams(n_boot=300,min_test_events=50)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'02'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_events.py'),str(ROOT/'tests'/'test_signed_jumps.py')],
                       cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(se.registration(ep),indent=1))

................                                                         [100%]
16 passed in 54.61s


{
 "horizons": [
  1,
  2,
  3,
  5,
  10,
  15,
  30,
  60
 ],
 "cluster_gap": 5,
 "excursion_window": 15,
 "cost_primary_bp": 12.0,
 "cost_grid_bp": [
  7.0,
  12.0,
  14.0
 ],
 "n_boot": 2000,
 "min_test_events": 200,
 "status": "PREREGISTERED_SJ003",
 "version": "SJ02a-0.1.0",
 "selection": "argmax DEV net 95% lower bound over horizon x {continuation, reversal} x {all, top, bottom pi_c quintile}",
 "gates": {
  "NET_EDGE": "TEST net lower 95% > 0",
  "GROSS_EDGE_ONLY": "TEST gross lower 95% > 0, net not",
  "NO_EDGE": "otherwise"
 }
}


## SJ 01 engine (unchanged) and fitted signal

In [2]:
T0=time.time()
if SYNTHETIC:
    S={a:s for a,s in zip(p.assets,sj.simulate(n_days=50,n_assets=len(p.assets),eta=((0.2,0.05),(0.05,0.3)),half_life=1,seed=5,df=8,base_rate=1e-3).values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
assets,pan,dev_end=sj.prepare(S,p)
print('DEV/TEST boundary',dev_end,f'{time.time()-T0:.0f}s')
if SYNTHETIC:
    f0,_=sj.fit_ladder(assets,pan,p,models=('e',));rows=[]
    for (m,s),f in f0.items():
        r={'model':m,'sign':s,'half_life_min':f.half_life,'ll_dev':f.ll_dev,'n_dev':f.n_dev,'converged':f.converged}
        r.update(dict(zip(sj.design_names(pan,m,p),f.b)));r.update({f'eta{k}':v for k,v in enumerate(f.eta)});rows.append(r)
    tab=pd.DataFrame(rows)
else:
    tab=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_fits.csv')
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
fits=se.fits_from_table(tab,pan,p)
lam=se.intensities(fits,assets,pan)
ev=se.find_events(pan,lam,ep.cluster_gap)
print(f'{len(ev)} jumps, {ev.starter.sum()} cluster starters; pi_c quantiles:',ev.pi_c.quantile([.05,.2,.5,.8,.95]).round(3).tolist())

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


DEV/TEST boundary 2026-02-15 00:00:00+00:00 100s
jump counts reproduce SJ 01
18580 jumps, 15495 cluster starters; pi_c quantiles: [0.583, 0.597, 0.61, 0.7, 0.705]


## Perp paths

In [3]:
if SYNTHETIC:
    perp=se.plant_drift(S,assets,0.0,10)
else:
    perp={A:sj.load_spot_1m(BIN/f'{A}USDT_um_1m.parquet') for A in p.assets}
ev=se.attach_paths(ev,perp,ep)
cov=ev.groupby('asset')[[f'g_imm_{k}' for k in ep.horizons]].apply(lambda d:d.notna().mean()).round(3)
print('share of events with a perp path:');display(cov)
ev.to_csv(OUT/'02_events.csv.gz',index=False)
cuts=se.quintile_cuts(ev);print('pi_c DEV quintile cuts (20%, 80%):',np.round(cuts,4))

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps_events.py:145: RuntimeWarning: All-NaN slice encountered
  ev.loc[grp.index, "mfe_15"] = np.nanmax(rel, axis=1)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps_events.py:146: RuntimeWarning: All-NaN slice encountered
  ev.loc[grp.index, "mae_15"] = np.nanmin(rel, axis=1)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps_events.py:145: RuntimeWarning: All-NaN slice encountered
  ev.loc[grp.index, "mfe_15"] = np.nanmax(rel, axis=1)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps_events.py:146: RuntimeWarning: All-NaN slice encountered
  ev.loc[grp.index, "mae_15"] = np.nanmin(rel, axis=1)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps_events.py:145: RuntimeWarning: All-NaN slice encountered
  ev.loc[grp.index, "mfe_15"] = np.nanmax(rel, axis=1)
D:\Projects\deribit_market_data\src\data_engineering\modeling\sig

share of events with a perp path:


,g_imm_1,g_imm_2,g_imm_3,g_imm_5,g_imm_10,g_imm_15,g_imm_30,g_imm_60
asset,,,,,,,,
BTC,0.913,0.913,0.913,0.913,0.913,0.913,0.913,0.913
ETH,0.921,0.921,0.921,0.921,0.921,0.921,0.921,0.921
SOL,0.926,0.926,0.926,0.926,0.926,0.926,0.926,0.926
XRP,0.900,0.900,0.900,0.900,0.900,0.900,0.900,0.900


pi_c DEV quintile cuts (20%, 80%): [0.5971 0.7004]


## DEV surface → rule; TEST verdict

In [4]:
surf=pd.concat([se.surface(ev,ep,cuts,per,mode) for per in ('dev','test') for mode in ('imm','del')],ignore_index=True)
surf.to_csv(OUT/'02_surface.csv',index=False)
rule=se.select_rule(surf[(surf.period=='dev')&(surf.entry=='imm')],ep)
print('selected rule (DEV):',rule)
res_imm=se.evaluate(ev,rule,cuts,ep,'imm');res_del=se.evaluate(ev,rule,cuts,ep,'del')
evals=pd.DataFrame([{k:v for k,v in r.items() if k!='rule'} for r in (res_imm,res_del)])
display(evals.round(2));evals.to_csv(OUT/'02_test_eval.csv',index=False)
show=surf[(surf['filter']=='all')&(surf.direction=='continuation')].pivot_table(index='horizon_min',columns=['period','entry'],values='gross_bp')
print('Mean continuation return, all cluster starters (bp):');display(show.round(2))

selected rule (DEV): {'filter': 'all', 'horizon_min': 1, 'direction': 'reversal', 'dev_gross_bp': 1.7281402669625505, 'dev_gross_lo': 0.7848927800705228, 'dev_n': 8560}


,entry,test_n,test_days,gross_bp,gross_lo,gross_hi,net_7bp,net_7bp_lo,net_12bp,net_12bp_lo,net_14bp,net_14bp_lo,verdict
0,imm,5679,239,0.13,-0.63,0.87,-6.87,-7.63,-11.87,-12.63,-13.87,-14.63,NO_EDGE
1,del,5679,239,-0.44,-1.11,0.23,-7.44,-8.11,-12.44,-13.11,-14.44,-15.11,NO_EDGE


Mean continuation return, all cluster starters (bp):


period        dev        test      
entry         del   imm   del   imm
horizon_min                        
1            0.42 -1.73  0.44 -0.13
2            0.40 -1.31  0.99  0.31
3            0.50 -1.33  0.94  0.86
5            0.47 -1.49  0.83  0.61
10           0.66 -1.07  0.22  0.16
15           1.17 -0.19  0.25  0.31
30           2.38  0.98  0.05  0.14
60           2.08  0.53 -0.43 -0.34

## Shape of the move (TEST, descriptive)

In [5]:
car=se.car_curve(ev,ep,cuts,'imm');car.to_csv(OUT/'02_car_curve.csv',index=False)
display(car.pivot_table(index=['sign','pi_c_quintile'],columns='horizon_min',values='mean_bp').round(2))
t=ev[ev.starter&~ev.dev]
exc=t.groupby(['asset','sign'])[['mfe_15','mae_15','jump_bp']].agg(['mean','median']).round(2)
display(exc);exc.to_csv(OUT/'02_excursions.csv')
allj=pd.concat([se.surface(ev.assign(starter=True),ep,cuts,'test','imm').assign(events='all_jumps')])
allj.to_csv(OUT/'02_surface_all_jumps.csv',index=False)
display(allj[(allj['filter']=='all')&(allj.direction=='continuation')][['horizon_min','gross_bp','gross_lo','gross_hi','n']].round(2))

horizon_min            1     2     3     5      10     15     30     60
sign pi_c_quintile                                                     
down 0               0.14  0.54  0.40  0.42  -0.24  -1.58  -2.71  -4.07
     1              -0.60  0.02  1.17  0.59  -0.02   0.54  -0.64  -2.90
     2              -0.53 -0.45  2.39  4.07   4.86   4.59   3.50   1.91
up   0               7.51  1.21 -4.99  6.80  -6.96   0.20  -1.56   6.05
     1              22.11  2.73  3.61 -3.77 -44.41 -38.34 -16.48  36.30
     2               0.23  2.13  2.61  1.32   0.57   1.09   2.41   2.85
     3              -0.63 -0.52  0.01  0.51   0.66   1.27   2.57   4.37
     4               0.43  0.25  0.15 -0.83  -1.67  -0.99  -1.30  -1.68

mfe_15        mae_15        jump_bp       
             mean median   mean median    mean median
asset sign                                           
BTC   -1    16.43   7.97 -14.26 -10.00  -22.95 -19.53
       1    16.45   7.24 -14.24 -10.04   24.53  19.92
ETH   -1    27.18  12.89 -21.28 -15.83  -35.89 -31.89
       1    25.57  10.20 -22.12 -17.41   38.45  34.13
SOL   -1    27.99  15.26 -24.64 -17.68  -39.83 -34.56
       1    26.91  11.62 -25.24 -20.05   41.90  36.14
XRP   -1    22.87  11.35 -22.56 -15.29  -35.95 -29.20
       1    23.66  11.95 -21.26 -15.51   35.30  30.53

,horizon_min,gross_bp,gross_lo,gross_hi,n
0,1,0.10,-0.86,1.17,6689
2,2,0.69,-0.66,2.30,6689
4,3,0.93,-0.61,2.80,6689
6,5,0.83,-0.74,2.50,6689
8,10,0.67,-1.12,2.51,6689
10,15,1.06,-1.35,3.51,6689
12,30,0.53,-2.13,3.18,6689
14,60,0.12,-3.15,3.43,6689


## Decision

In [6]:
dec={'version':se.VERSION,'status':ep.status,'rule':rule,'verdict':res_imm['verdict'],
     'test_gross_bp':res_imm['gross_bp'],'test_gross_lo':res_imm['gross_lo'],
     'test_net_12bp':res_imm['net_12bp'],'test_net_12bp_lo':res_imm['net_12bp_lo'],
     'test_net_7bp':res_imm['net_7bp'],'test_net_7bp_lo':res_imm['net_7bp_lo'],
     'delayed_entry':{k:res_del[k] for k in ('gross_bp','gross_lo','net_12bp','net_12bp_lo','verdict')},
     'test_events':res_imm['test_n'],'build_SJ02b':res_imm['verdict']=='NET_EDGE','synthetic':SYNTHETIC}
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'02_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'02_manifest.json').write_text(json.dumps({'registration':se.registration(ep),'runtime_s':time.time()-T0,
        'events':int(len(ev)),'starters':int(ev.starter.sum())},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "SJ02a-0.1.0",
 "status": "PREREGISTERED_SJ003",
 "rule": {
  "filter": "all",
  "horizon_min": 1,
  "direction": "reversal",
  "dev_gross_bp": 1.7281402669625505,
  "dev_gross_lo": 0.7848927800705228,
  "dev_n": 8560
 },
 "verdict": "NO_EDGE",
 "test_gross_bp": 0.13123054524905006,
 "test_gross_lo": -0.6256161595245382,
 "test_net_12bp": -11.868769454750948,
 "test_net_12bp_lo": -12.625616159524537,
 "test_net_7bp": -6.86876945475095,
 "test_net_7bp_lo": -7.6256161595245375,
 "delayed_entry": {
  "gross_bp": -0.44081532517561667,
  "gross_lo": -1.1091694389952014,
  "net_12bp": -12.440815325175617,
  "net_12bp_lo": -13.109169438995202,
  "verdict": "NO_EDGE"
 },
 "test_events": 5679,
 "build_SJ02b": false,
 "synthetic": false
}
total 110s
